# Notebook14 bounded study definitions
Purpose: run the four-condition full-JEPA screen across two GPUs under one original7200second deadline. Inputs: tagged definitions, pinned train/development memberships, short validation receipts. Outputs:16training runs,16fixedepoch20probes, CSV/HTML/PNG decision report. Runtime: up to two hours after launch. Fullstate checkpoints: checkpoints/<run_id>/stages/worker{0,1}/. Resume: unchanged Notebook14 Run All before original deadline; invalid manifests or changed sources raise. One writer per worker and coordinator; test set remains closed.

In [ ]:
"""Two-GPU, four-pair, two-hour checkpointed JEPA screening orchestration."""
from pathlib import Path
from datetime import datetime,timezone
import copy,gc,hashlib,json,os,shlex,statistics,subprocess,sys,time,types
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json,digest_file,run_writer
MAIN='notebooks/14_Routing_Capacity_Screen.ipynb'
SEEDS=list(range(20261601,20261605))
VARIANTS=['vit','normalized_kernel','compensated_kernel','compensated_wide_kernel']

_MODULES={}
def _load(root,name):
    path=Path(root)/'experiments/14_routing_capacity'/(name+'.ipynb')
    key=str(path.resolve())
    if key in _MODULES:return _MODULES[key]
    module=types.ModuleType('fmm14_workflow_'+name);module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module);_MODULES[key]=module;return module

def _read(path):return json.loads(Path(path).read_text())
def _hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def _consumer_hash(root):return _hash([c.source for c in nbformat.read(Path(root)/MAIN,as_version=4).cells if c.cell_type=='code'])
def _exists(name):return subprocess.run(['tmux','has-session','-t',name],capture_output=True).returncode==0

def study_configuration(run_id='routing-capacity-001'):
    if not run_id or any(not(c.isalnum()or c in '-_')for c in run_id):raise ValueError('Safe runID required')
    return {'schema':'notebook14-screening-study/v1','run_id':run_id,'hard_limit_seconds':7200,
      'seeds':SEEDS,'variants':VARIANTS,'devices':['cuda:0','cuda:1'],'update_candidates':[600,450,350],
      'round_updates':50,'probe_endpoints':['final'],'probe_epochs':20,'image_size':128,'effective_batch':128,
      'analysis_role':'Exploratory four-pair screening; no robust5percent significance claim',
      'selection':'Timing-only update selection before scientific training; no quality stopping or seed selection',
      'calibration':'First-block wide/compensated6 entropy matched using4training images and discardedseed20269914; spatialterm unchanged',
      'heldout_status':'NOT_RUN','official_test_pixels_read':0,'automatic_12h_followup':False}

def _status(artifact,stage,status,**details):
    record={'status':status,'stage':stage,'updated_unix':time.time(),**details};atomic_json(Path(artifact)/'study_status.json',record);return record

def _receipts(root):
    paths=['artifacts/notebook_14_validation/encoder_validation.json',
           'artifacts/notebook_14_validation/runner-v3/runner_validation.json',
           'artifacts/notebook_14_validation/analysis_validation.json',
           'artifacts/notebook_14_validation/pipeline/validation_result.json']
    records={}
    for relative in paths:
        path=Path(root)/relative
        if not path.exists():raise RuntimeError('Required short validation missing: '+relative)
        record=_read(path)
        if record.get('status')!='PASS':raise RuntimeError('Required validation notPASS: '+relative)
        records[relative]={'sha256':digest_file(path),'status':'PASS'}
    current=_load(root,'runner').stage_definition_hashes(root)
    encoder=_read(Path(root)/paths[0])
    if encoder['source_export_sha256']!=current['experiments/14_routing_capacity/encoder.ipynb']:raise ValueError('Encoder validation is stale')
    runner=_read(Path(root)/paths[1])
    for name,expected in runner['source_hashes'].items():
        if current['experiments/14_routing_capacity/'+name+'.ipynb']!=expected:raise ValueError('Runner validation is stale: '+name)
    analysis=_read(Path(root)/paths[2])
    if analysis['analysis_sha256']!=digest_file(Path(root)/'experiments/14_routing_capacity/analysis.ipynb'):raise ValueError('Analysis validation is stale')
    if _read(Path(root)/paths[3])['source_hashes']!=current:raise ValueError('Pipeline validation is stale')
    return records

def _verify(root,artifact,binding):
    if _load(root,'runner').stage_definition_hashes(root)!=binding['source_hashes']:raise ValueError('Frozen experiment source exports changed')
    if _consumer_hash(root)!=binding['consumer_notebook_sha256']:raise ValueError('Frozen consumer code changed')
    if _read(Path(artifact)/'budget_clock.json')!=binding['budget_clock']:raise ValueError('Original deadline changed')
    if digest_file(Path(root)/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')!=binding['base_reference_plan_sha256']:raise ValueError('Base reference plan changed')
    if binding['configuration']!=study_configuration(binding['configuration']['run_id']):raise ValueError('Study design changed')

def readiness(root,configuration=None):
    root=Path(root);config=configuration or study_configuration();receipts=_receipts(root)
    import torch
    if torch.cuda.device_count()<2:raise RuntimeError('This prospectively paired screen requires two local GPUs')
    return {'status':'PASS','configuration':config,'validation':receipts,'consumer_notebook':MAIN,
      'checkpoint_root':str(root/'checkpoints'/config['run_id']),'protocol':'Four conditions ×four freshpairedseeds, fixedepoch20 finalprobes; twohourabsolute cap, no12hlaunch.'}

def _bootstrap(root):
    return next(c.source for c in nbformat.read(Path(root)/MAIN,as_version=4).cells if c.get('id')=='bootstrap-routing-screen'and'ahr-export'in c.metadata.get('tags',[]))

def _spawn(root,artifact,session,expression,notebook):
    root,artifact,notebook=Path(root),Path(artifact),Path(notebook)
    code=_bootstrap(root)+'\n'+expression+'\n';notebook.parent.mkdir(parents=True,exist_ok=True)
    if notebook.exists():
        if [c.source for c in nbformat.read(notebook,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
    else:
        nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook14 detached worker\nPurpose: execute the frozen registered screen. Inputs: source-bound parent plan/data. Outputs: metrics and full-state checkpoints. Runtime: remaining original two-hour budget. Resume with unchanged Notebook14 Run All; latest valid checkpoint is mandatory.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),notebook)
    if _exists(session):return
    command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(notebook)]
    shell='exec '+shlex.join(command)+' > '+shlex.quote(str(notebook.with_suffix('.log')))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)

def launch_study(root,configuration=None):
    root=Path(root).resolve();config=configuration or study_configuration();artifact=root/'artifacts/notebook_14'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        binding_path=artifact/'study_binding.json'
        if binding_path.exists():
            binding=_read(binding_path);_verify(root,artifact,binding)
            prior=_read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if prior.get('status')=='COMPLETED':return prior
            if time.time()>=binding['budget_clock']['deadline_unix']:raise TimeoutError('Original two-hour deadline expired; cannot silently restart')
        else:
            ready=readiness(root,config)
            if (artifact/'budget_clock.json').exists()or(artifact/'plan.json').exists()or(artifact/'workers').exists():raise ValueError('Unbound existing run; cannot restart')
            prep=_load(root,'preparation');base=prep.base_config(root)
            bundle=prep.prepare_bundle(root,artifact/'data',base,verify_images=True)
            entropy=prep.entropy_calibration(root,artifact,bundle)
            now=time.time();clock={'started_unix':now,'deadline_unix':now+7200,'max_seconds':7200,
              'policy':'Original absolute clock includes discardedtiming, exactresume preflight, training, evaluation, reporting and recovery; never resets.'}
            binding={'schema':'notebook14-study-binding/v1','configuration':config,'source_hashes':_load(root,'runner').stage_definition_hashes(root),
              'consumer_notebook_sha256':_consumer_hash(root),'budget_clock':clock,'validation':ready['validation'],
              'dataset_fingerprint':bundle['dataset'].fingerprint,'split_sha256':bundle['split_manifest']['split_sha256'],
              'entropy_calibration':entropy,'base_reference_plan_sha256':digest_file(root/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')}
            atomic_json(binding_path,binding);atomic_json(artifact/'budget_clock.json',clock)
        session='fmm-routing-screen-'+config['run_id'];watch='fmm-routing-deadline-'+config['run_id']
        if not _exists(session):_status(artifact,'launch_or_exact_resume','RUNNING',deadline_unix=binding['budget_clock']['deadline_unix'])
        _spawn(root,artifact,watch,'WORKFLOW.run_watchdog(ROOT, '+repr(config['run_id'])+')',artifact/'deadline_watchdog.ipynb')
        _spawn(root,artifact,session,'WORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')',artifact/'coordinator.ipynb')
        launch={'status':'LAUNCHED','session':session,'worker_sessions':['fmm-routing-'+config['run_id']+'-worker0','fmm-routing-'+config['run_id']+'-worker1'],
          'artifact':str(artifact),'deadline_unix':binding['budget_clock']['deadline_unix'],
          'deadline_utc':datetime.fromtimestamp(binding['budget_clock']['deadline_unix'],timezone.utc).isoformat(),
          'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','resume':'Run All in unchanged '+MAIN+' before originaldeadline'}
        atomic_json(artifact/'launch.json',launch);return launch

def _await_jobs(root,artifact,binding,jobs,stage):
    while True:
        _verify(root,artifact,binding)
        if time.time()>=binding['budget_clock']['deadline_unix']:raise TimeoutError('Original two-hour deadline reached')
        done=[j['result'].exists()for j in jobs]
        if all(done):return [_read(j['result'])for j in jobs]
        for j,complete in zip(jobs,done):
            if not complete and not _exists(j['session']):raise RuntimeError('Detached '+stage+' job exited without completed evidence: '+j['session'])
        details={}
        for j in jobs:
            progress=j.get('progress')
            if progress and progress.exists():details[j['session']]=_read(progress)
        _status(artifact,stage,'RUNNING',completed_workers=sum(done),total_workers=len(jobs),workers=details,
          elapsed_seconds=time.time()-binding['budget_clock']['started_unix'],deadline_unix=binding['budget_clock']['deadline_unix'])
        time.sleep(10)

def _choose_updates(root,binding,calibrations):
    deadline=binding['budget_clock']['deadline_unix'];remaining=deadline-time.time();choices=[]
    history=_read(Path(root)/'artifacts/notebook_13/geometry-ablation-001/stages/learning/stage_evaluation.json')
    probe_seconds=(history['completed_unix']-history['started_unix'])/len(history['evaluations'])
    slowest=max(sum(r['median_step_seconds']for r in c['rows'])for c in calibrations)
    overhead=max(sum(r['round_overhead_seconds']for r in c['rows'])for c in calibrations)
    eval_reserve=max(600.,1.5*8*probe_seconds)+120.
    for updates in [600,450,350]:
        rounds=math_ceil(updates/50)
        train=1.15*2*(updates*slowest+rounds*overhead)
        total=train+eval_reserve+360.
        choices.append({'updates':updates,'predicted_training_seconds_per_worker':train,'predicted_total_remaining_seconds':total})
    selected=next((c for c in choices if c['predicted_total_remaining_seconds']<=remaining),None)
    if selected is None:raise TimeoutError('No complete fourpairdesign fits originaldeadline: '+str(choices))
    return {**selected,'evaluation_reserve_seconds':eval_reserve,'preflight_recovery_seconds':360.,'remaining_seconds':remaining,
      'historical_seconds_per_probe':probe_seconds,'timing_only_candidates':choices,'quality_selection':False,
      'policy':'Highest600/450/350 updatecandidate fitting slowestGPU plus15percentmargin, measuredroundoverhead,8probe reserve perGPU andpreflight.'}

def math_ceil(x):return int(-(-x//1))

def _plan(root,binding,budget,bundle):
    prep=_load(root,'preparation');base=prep.base_config(root);wide=binding['entropy_calibration']['wide_content_length_factor'];arms=[]
    for index,seed in enumerate(SEEDS):
        worker=index//2
        for variant in VARIANTS:
            cfg=prep.arm_config(base,variant,wide);cfg['device']='cuda:'+str(worker)
            cfg['seeds'].update(model_init=seed,data_order=974101+1000*index,augmentation=975202+1000*index,mask=976303+1000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000;cfg['training']['updates']=budget['updates'];cfg['evaluation']['diagnostic_steps']=[budget['updates']]
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit'if variant=='vit'else'semantic','variant':variant,'condition':variant,'seed':seed,'worker':worker,'config':cfg})
    return {'schema':'notebook14-stage-plan/v1','run_id':binding['configuration']['run_id'],'stage_id':'learning','stage_role':'learning',
      'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':budget['updates'],'milestones':[budget['updates']],
      'round_updates':50,'probe_endpoints':['final'],'hard_limit_seconds':7200,'effective_batch':128,'seeds':SEEDS,'registered_seeds':SEEDS,
      'variants':VARIANTS,'arms':arms,'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
      'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
      'resume_notebook':MAIN,'resume_cell':'launch-routing-screen','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
      'study_configuration_hash':_hash(binding['configuration']),'numerical_backend':'exact-dense-routing-capacity-v1','heldout_status':'NOT_RUN',
      'budget_selection':budget,'entropy_calibration':binding['entropy_calibration'],
      'analysis':{'role':'exploratory_screening','n_pairs':4,'df':3,'multiplicity':'Holm','family_alpha':.05},
      'primary_analysis':{'role':'exploratory_only','n_pairs':4,'df':3,'alpha':.05,'multiplicity':'Holm',
         'family':['compensated_kernel_minus_vit_final','compensated_wide_kernel_minus_vit_final']},
      'selection':'Fixedfinal checkpoint andepoch20 probe; reuseddevelopment2020; officialtest unopened.',
      'capacity_policy':'Sharedscaffold/MLPs/norms/predictor, sharedkernelvalue/output/bandwidth/mixture andfirst6coordinates/head; wideextra26active. Equalupdatesnotcompute.',
      'failure_policy':'Stopfirstnumericalfailure; no batchskip; invalidmanifest neverrestarts.',
      'excluded_pilot_seeds':list(range(20261501,20261509))+list(range(20261401,20261407))}

def run_study(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_14'/run_id;binding=_read(artifact/'study_binding.json');_verify(root,artifact,binding)
    with run_writer(artifact/'coordinator_lock'):
        try:
            if (artifact/'study_status.json').exists()and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
            prep=_load(root,'preparation');jobs=[]
            for device in binding['configuration']['devices']:
                suffix=device.replace(':','-');session='fmm-routing-'+run_id+'-calibration-'+suffix
                path=artifact/'calibration'/suffix;result=path/'result.json'
                if not result.exists():_spawn(root,artifact,session,'WORKFLOW._load(ROOT, "preparation").run_calibration(ROOT, '+repr(run_id)+', '+repr(device)+')',path/'worker.ipynb')
                jobs.append({'session':session,'result':result,'progress':path/'progress.json'})
            calibrations=_await_jobs(root,artifact,binding,jobs,'discarded_timing')
            if any(c.get('status')!='PASS'or c.get('source_hashes')!=binding['source_hashes']for c in calibrations):raise ValueError('Discardedtiming evidence source identity differs')
            bundle=prep.prepare_bundle(root,artifact/'data',prep.base_config(root),verify_images=True)
            if (artifact/'plan.json').exists():plan=_read(artifact/'plan.json')
            else:plan=_plan(root,binding,_choose_updates(root,binding,calibrations),bundle);prep.immutable(artifact/'plan.json',plan)
            if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Frozen plan source identity differs')
            _load(root,'analysis').register_analysis(root,artifact,plan)
            jobs=[]
            for worker in range(2):
                stage='worker'+str(worker);folder=artifact/'workers'/stage;folder.mkdir(parents=True,exist_ok=True)
                child=copy.deepcopy(plan);child['stage_id']=stage;child['stage']['id']=stage
                child['seeds']=SEEDS[worker*2:worker*2+2];child['arms']=[a for a in plan['arms']if a['worker']==worker]
                child['parent_plan_hash']=_hash(plan);prep.immutable(folder/'plan.json',child)
                session='fmm-routing-'+run_id+'-'+stage;result=folder/'worker_result.json'
                if not result.exists():_spawn(root,artifact,session,'WORKFLOW.run_worker(ROOT, '+repr(run_id)+', '+str(worker)+')',folder/'worker.ipynb')
                jobs.append({'session':session,'result':result,'progress':folder/'stage_status.json'})
            results=_await_jobs(root,artifact,binding,jobs,'paired_training_and_fixed_probes')
            if any(r.get('status')!='COMPLETE'for r in results):raise RuntimeError('Worker did not complete registered endpoints')
            evaluations={};training={};chart_rows=[]
            for worker,result in enumerate(results):
                stage=artifact/'workers'/('worker'+str(worker));e=_read(stage/'stage_evaluation.json');t=_read(stage/'training_result.json')
                if set(evaluations)&set(e['evaluations']):raise ValueError('Duplicate model endpoints acrossworkers')
                evaluations.update(e['evaluations']);training.update(t['arms']);chart_rows.extend(e.get('chart_rows',[]))
            if len(evaluations)!=16 or len(training)!=16:raise AssertionError('Every fourcondition/fourseedendpoint required')
            combined={'status':'COMPLETE','run_id':run_id,'stage_id':'learning','evaluations':evaluations,'chart_rows':chart_rows,
              'membership':_read(artifact/'workers/worker0/stage_evaluation.json')['membership'],'official_test':'NOT_RUN',
              'image_size':128,'pretraining_images':len(bundle['train_ids']),'probe_selection':'fixed_final_epoch20','labels_in_self_supervised_loss':False}
            trained={'status':'COMPLETE','planned_updates':plan['updates'],'arms':training,'equal_updates':True,'paired_traces_within_seed':True,'no_failed_attempts':True}
            atomic_json(artifact/'stage_evaluation.json',combined);atomic_json(artifact/'training_result.json',trained)
            report=_load(root,'analysis').build_report(root,artifact,plan,combined,trained)
            atomic_json(artifact/'study_results.json',report)
            _verify(root,artifact,binding)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-binding['budget_clock']['started_unix'],updates=plan['updates'],
              seeds=SEEDS,variants=VARIANTS,completed_models=16,heldout_status='NOT_RUN',report=report)
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE'if isinstance(error,TimeoutError)else'FAILED',error=repr(error),
              elapsed_seconds=time.time()-binding['budget_clock']['started_unix'],conclusion='UNKNOWN until all registered endpoints complete')
            raise

def run_worker(root,run_id,worker):
    root=Path(root);artifact=root/'artifacts/notebook_14'/run_id;binding=_read(artifact/'study_binding.json');_verify(root,artifact,binding)
    stage='worker'+str(worker);folder=artifact/'workers'/stage;plan=_read(folder/'plan.json');deadline=binding['budget_clock']['deadline_unix']
    prep,runner,evaluation=(_load(root,n)for n in('preparation','runner','evaluation'))
    bundle=prep.prepare_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=True)
    with run_writer(folder/'worker_lock'):
        try:
            _status(folder,'exact_resume_preflight','RUNNING')
            runner.prepare_stage(root,folder,plan,bundle)
            preflight_path=folder/'preflight_result.json'
            if not preflight_path.exists():atomic_json(preflight_path,runner.verify_stage_resume(root,folder,plan,bundle,deadline))
            if _read(preflight_path).get('status')!='PASS':raise RuntimeError('Actualfullstateexactresume didnotpass')
            eval_preflight=folder/'evaluation_preflight_result.json'
            if not eval_preflight.exists():atomic_json(eval_preflight,evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline))
            if _read(eval_preflight).get('status')!='PASS':raise RuntimeError('Probeexactresume/integrationfailed')
            _verify(root,artifact,binding);_status(folder,'paired_training','RUNNING',updates=plan['updates'])
            tpath=folder/'training_result.json'
            if tpath.exists()and _read(tpath).get('status')=='COMPLETE':
                training=_read(tpath);evaluation._stage_jobs(plan,training)
                from fmm_jepa.food101_training import validate_training_manifest
                for arm in plan['arms']:
                    state=validate_training_manifest(root/'checkpoints'/run_id/'stages'/stage/'arms'/arm['id'])
                    if state['successful_updates']!=plan['updates']or state['attempted_updates']!=plan['updates']:raise ValueError('Completedtrainingstate differs')
                    runner.validate_stage_milestones(root/'checkpoints'/run_id/'stages'/stage/'arms'/arm['id'],plan,required=plan['milestones'])
            else:training=runner.train_stage(root,folder,plan,bundle,deadline);atomic_json(tpath,training)
            _status(folder,'frozen_feature_evaluation','RUNNING')
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            if result.get('status')!='COMPLETE':raise RuntimeError('Incomplete endpoint evaluation')
            _verify(root,artifact,binding)
            record={'status':'COMPLETE','stage':stage,'completed_models':len(result['evaluations']),'updates':plan['updates'],'updated_unix':time.time()}
            atomic_json(folder/'worker_result.json',record);_status(folder,'complete','COMPLETE',**{k:v for k,v in record.items()if k not in('status','stage')});return record
        except BaseException as error:
            _status(folder,'interrupted','FAILED',error=repr(error));raise

def run_watchdog(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_14'/run_id;binding=_read(artifact/'study_binding.json');deadline=binding['budget_clock']['deadline_unix']
    old=_load13watchdog(root);started=time.time();last_hourly=started
    while True:
        prior=_read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if prior.get('status')in('COMPLETED','FAILED','INCOMPLETE'):
            if prior['status']!='COMPLETED':
                sessions=['fmm-routing-'+run_id+'-worker'+str(i)for i in range(2)]+['fmm-routing-'+run_id+'-calibration-cuda-'+str(i)for i in range(2)]
                for session in sessions:
                    if _exists(session):old._terminate_session_tree(session)
            atomic_json(artifact/'watchdog_result.json',{'status':'TERMINAL','study_status':prior['status'],'updated_unix':time.time()});return
        if time.time()-started>60 and not _exists('fmm-routing-screen-'+run_id):
            sessions=['fmm-routing-'+run_id+'-worker'+str(i)for i in range(2)]+['fmm-routing-'+run_id+'-calibration-cuda-'+str(i)for i in range(2)]
            for session in sessions:
                if _exists(session):old._terminate_session_tree(session)
            _status(artifact,'coordinator_disappeared','INCOMPLETE',conclusion='UNKNOWN: coordinator exited without a completed registered result')
            return
        if time.time()-last_hourly>=3600:
            from fmm_jepa.food101_training import validate_training_manifest
            try:
                _verify(root,artifact,binding);checked=[]
                for manifest in sorted((root/'checkpoints'/run_id/'stages').glob('*/arms/*/manifest.json')):
                    state=validate_training_manifest(manifest.parent)
                    if state['attempted_updates']!=state['successful_updates']:raise ValueError('Failed numerical update encountered')
                    checked.append({'manifest':str(manifest),'successful_updates':state['successful_updates']})
                atomic_json(artifact/('hourly_review_'+str(int(time.time()))+'.json'),{'status':'PASS','checked':checked,'updated_unix':time.time(),'deadline_unix':deadline,'scope':'Source, originalclock and actual resumable checkpoints; no quality-based intervention'})
                last_hourly=time.time()
            except BaseException as error:
                _status(artifact,'hourly_integrity_failure','FAILED',error=repr(error),conclusion='UNKNOWN');continue
        if time.time()>=deadline:
            sessions=['fmm-routing-'+run_id+'-worker'+str(i)for i in range(2)]+['fmm-routing-'+run_id+'-calibration-cuda-'+str(i)for i in range(2)]+['fmm-routing-screen-'+run_id]
            for session in sessions:
                if _exists(session):old._terminate_session_tree(session)
            _status(artifact,'deadline','INCOMPLETE',conclusion='UNKNOWN: missing endpoints at original hard cutoff',deadline_unix=deadline)
            atomic_json(artifact/'watchdog_result.json',{'status':'HARD_STOP','updated_unix':time.time(),'sessions':sessions});return
        atomic_json(artifact/'watchdog_heartbeat.json',{'status':'HEALTHY','updated_unix':time.time(),'deadline_unix':deadline});time.sleep(10)

def _load13watchdog(root):
    path=Path(root)/'experiments/13_product_geometry/deadline_watchdog.ipynb';module=types.ModuleType('fmm14_watchdog_base');module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module);_MODULES[key]=module;return module

def inspect_study(root,run_id='routing-capacity-001'):
    artifact=Path(root)/'artifacts/notebook_14'/run_id
    return {name:_read(artifact/name)for name in('study_status.json','budget_clock.json','launch.json','study_results.json')if(artifact/name).exists()}
